# Retail AI — YOLOv8 Training on SKU-110K
**Runtime: GPU (T4 recommended)**  
Go to `Runtime → Change runtime type → T4 GPU` before running.

**Steps:**
1. Set up Kaggle credentials
2. Download SKU-110K dataset
3. Prepare dataset (YOLO format)
4. Train YOLOv8
5. Evaluate and download best model

In [ ]:
# ── Cell 1: Check GPU ────────────────────────────────────────────
!nvidia-smi
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None"}')

In [ ]:
# ── Cell 2: Install packages ─────────────────────────────────────
!pip install ultralytics kaggle -q
from ultralytics import YOLO
print('ultralytics ready')

In [ ]:
# ── Cell 3: Kaggle credentials ───────────────────────────────────
# How to get your kaggle.json:
#   1. Go to https://www.kaggle.com → Account → API → Create New Token
#   2. Upload the downloaded kaggle.json using the Files panel (left sidebar)
import os, shutil
os.makedirs('/root/.kaggle', exist_ok=True)
shutil.copy('/content/kaggle.json', '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 0o600)
print('Kaggle credentials set.')

In [ ]:
# ── Cell 4: Download SKU-110K dataset ────────────────────────────
# Downloads annotations (~50MB) — images must be downloaded separately
# from original source if not included in Kaggle dataset
!kaggle datasets download -d thedatasith/sku110k-annotations -p /content/sku110k --unzip
!ls /content/sku110k

In [ ]:
# ── Cell 5: Prepare dataset ──────────────────────────────────────
import pandas as pd
import shutil
from pathlib import Path
from tqdm.notebook import tqdm
import yaml

DATASET_DIR = Path('/content/sku110k')
OUTPUT_DIR  = Path('/content/sku110k_yolo')
SUBSET      = 0  # 0 = use ALL images. Set e.g. 2000 for faster training.

def bbox_to_yolo(x1, y1, x2, y2, iw, ih):
    cx = max(0, min(1, (x1+x2)/(2*iw)))
    cy = max(0, min(1, (y1+y2)/(2*ih)))
    w  = max(0, min(1, (x2-x1)/iw))
    h  = max(0, min(1, (y2-y1)/ih))
    return cx, cy, w, h

def process_split(csv_path, img_src, out_img, out_lbl, subset):
    if not Path(csv_path).exists():
        print(f'Skipping {csv_path} — not found'); return 0
    df = pd.read_csv(csv_path, header=None,
                     names=['img','x1','y1','x2','y2','cls','iw','ih'])
    imgs = df['img'].unique()
    if subset: imgs = imgs[:subset]
    out_img.mkdir(parents=True, exist_ok=True)
    out_lbl.mkdir(parents=True, exist_ok=True)
    count = 0
    for name in tqdm(imgs):
        src = Path(img_src) / name
        if not src.exists(): continue
        shutil.copy2(src, out_img / name)
        rows = df[df['img'] == name]
        with open(out_lbl / (Path(name).stem + '.txt'), 'w') as f:
            for _, r in rows.iterrows():
                try:
                    cx,cy,w,h = bbox_to_yolo(r.x1,r.y1,r.x2,r.y2,r.iw,r.ih)
                    if w>0 and h>0: f.write(f'0 {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}\n')
                except: pass
        count += 1
    return count

splits_config = [
    ('train', SUBSET, DATASET_DIR/'annotations'/'annotations_train.csv', DATASET_DIR/'images'/'train'),
    ('val',   max(100, SUBSET//5) if SUBSET else 0, DATASET_DIR/'annotations'/'annotations_val.csv', DATASET_DIR/'images'/'val'),
    ('test',  max(100, SUBSET//5) if SUBSET else 0, DATASET_DIR/'annotations'/'annotations_test.csv', DATASET_DIR/'images'/'test'),
]
for split, sub, csv, img_src in splits_config:
    print(f'\n{split}:')
    n = process_split(csv, img_src, OUTPUT_DIR/'images'/split, OUTPUT_DIR/'labels'/split, sub)
    print(f'  {n} images prepared')

yaml_content = {
    'path': str(OUTPUT_DIR),
    'train': 'images/train', 'val': 'images/val', 'test': 'images/test',
    'nc': 1, 'names': ['object']
}
with open(OUTPUT_DIR/'data.yaml','w') as f: yaml.dump(yaml_content, f)
print(f'\ndata.yaml written to {OUTPUT_DIR}/data.yaml')

In [ ]:
# ── Cell 6: Train YOLOv8 on GPU ──────────────────────────────────
# Model options (larger = more accurate but slower):
#   yolov8n.pt  nano   — fastest, least accurate
#   yolov8s.pt  small  — good balance for T4 GPU
#   yolov8m.pt  medium — best accuracy on T4 (may be slow)

MODEL  = 'yolov8s.pt'   # recommended for Colab T4
EPOCHS = 50             # increase for better accuracy
IMGSZ  = 640            # full resolution
BATCH  = 16             # T4 can handle 16 comfortably

model = YOLO(MODEL)
results = model.train(
    data    = str(OUTPUT_DIR / 'data.yaml'),
    epochs  = EPOCHS,
    imgsz   = IMGSZ,
    batch   = BATCH,
    device  = 0,          # GPU 0
    name    = 'sku_retail_gpu',
    project = '/content/runs',
    patience= 10,
    plots   = True,
    workers = 2,
    cache   = True,       # cache images in RAM for faster epochs
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    flipud=0.0, fliplr=0.5, mosaic=1.0,
    degrees=5.0, translate=0.1, scale=0.5,
)
print('Training complete!')

In [ ]:
# ── Cell 7: Evaluate and show metrics ────────────────────────────
best = '/content/runs/sku_retail_gpu/weights/best.pt'
model = YOLO(best)
metrics = model.val(data=str(OUTPUT_DIR/'data.yaml'), split='test', plots=True)

box = metrics.box
f1  = 2 * box.mp * box.mr / (box.mp + box.mr + 1e-9)
print('\n=== EVALUATION RESULTS ===')
print(f'mAP@0.50       : {box.map50*100:.2f}%')
print(f'mAP@0.50:0.95  : {box.map*100:.2f}%')
print(f'Precision      : {box.mp*100:.2f}%')
print(f'Recall         : {box.mr*100:.2f}%')
print(f'F1 Score       : {f1*100:.2f}%')

In [ ]:
# ── Cell 8: Show training plots ──────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path

run_dir = Path('/content/runs/sku_retail_gpu')
plots = ['results.png', 'confusion_matrix.png', 'PR_curve.png', 'F1_curve.png']

fig, axes = plt.subplots(2, 2, figsize=(18, 12))
for ax, plot in zip(axes.flat, plots):
    p = run_dir / plot
    if p.exists():
        ax.imshow(mpimg.imread(str(p)))
        ax.set_title(plot.replace('.png',''), fontsize=12, fontweight='bold')
    ax.axis('off')
plt.tight_layout()
plt.savefig('/content/training_summary.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plots displayed and saved to /content/training_summary.png')

In [ ]:
# ── Cell 9: Download trained model to your PC ────────────────────
# This downloads best.pt (the trained weights) to your computer.
# Then copy it to retail-ai/backend/ and update .env:
#   YOLO_MODEL=best.pt

from google.colab import files
files.download('/content/runs/sku_retail_gpu/weights/best.pt')
files.download('/content/training_summary.png')
print('Download started. Copy best.pt to your retail-ai/backend/ folder.')
print('Then update backend/.env: YOLO_MODEL=best.pt')